# RFT Data: On-Policy 采样 → 高效通过筛选 → 关键 Turn loss mask

实现 `TASK.md` §2 阶段一的数据构造（不含训练）：

1. **采样（§2.1）**：在 Train Pool 57 题上跑当前 submission（base 模型、`temperature=0.7`、`max_tool_calls=28`、不设墙钟超时）。每个并发 slot 各有一个 Logging Proxy，记录发给 vLLM 的原始 `/v1/chat/completions` 请求和响应，以及 vLLM 实际使用的 `prompt_token_ids` / 生成的 `token_ids`。
   - 自适应 K：第一轮 K=2。2/2 的题，以及 0/2 但已编辑到 gold 区域的题，各补 2 条。
2. **高效通过筛选（§2.2，已修订）**：目标是减少步数，所以不只看“有时做对”，而看“有时高效地做对”。
   - 部分通过的题和全部通过的题都可以入选。
   - 正样本必须：通过、补丁碰到参考文件（`GOOD_LOCS`）、首次源码编辑 ≤ 第 12 次调用；全部通过的题还要求总调用 ≤ `ALL_PASS_CALLS_LIMIT`。
   - 每题按总调用从少到多取，全部通过的题最多 `MAX_PER_TASK_ALL_PASS` 条，部分通过的题最多 `MAX_PER_TASK_SOMETIMES` 条。
3. **自采样审计**：标签可信度、与 dev60 基线的分布对照、时间残留、调用计数差异。
4. **关键 turn loss mask（§2.3，已修订）**：loss 窗口仍是“首次读到 gold 文件之后”到“`submit_patch`”，但窗口内以下 turn 不计 loss：
   - 文件工具报错、参数被拒的 turn
   - 只调 `get_status` 的 turn
   - 编辑了最终补丁里没有的文件
   - 与之前完全相同、且期间没有源码编辑的重复调用
   - 最后一次有效编辑之后、超过 `POST_EDIT_KEEP` 个的验证 turn
   - 每个 loss turn 单独成一条序列：历史里的前几轮生成会被重新渲染，前缀永远接不上。

**挂载**（与 dev notebook 相同）：
- 竞赛数据
- `google/gemma-4` 模型（`gemma-4-31b-it-qat-w4a16-ct`）
- wheelhouse
- `tianyuxu2/gemma-4-extra-test-wheels`
- 你的 submission 数据集（不要带 LoRA adapter，否则采到的就不是 base 策略）

Accelerator 选 **GPU L4x4**，Internet **off**。

**运行**：用 *Save Version → Save & Run All (Commit)*。建议先把 `TRAIN_IDS = TRAIN_IDS[:2]`、`K_FIRST = 1`、`K_EXTRA = 0` 小跑一次，确认以下两项后再全量：
- token ids 已拿到
- 模板一致率为 100%

**续跑**：把上一版本的 Output 挂上来，并设置 `RESUME_DIR`。只重建数据、不采样时，设置 `SKIP_SAMPLING = True`（无需 GPU，但仍要挂模型以加载 tokenizer）。

**输出**（Output 标签页）：
- `rft_data_<RUN_NAME>.zip`：可直接用于训练
  - `train.jsonl`：每行一条序列，含 `input_ids`、`labels`（未选中位置为 -100）
  - `train_readable.jsonl`
  - `selection.json`
  - `stats.json`
  - `report.md`
- `rft_samples_<RUN_NAME>.zip`：全部原始样本（代理日志、trace、patch、测试日志），用于续跑和排查

## 0. Run Config

In [ ]:
import time

NOTEBOOK_START = time.time()

# ================================ RUN CONFIG ================================
# Outputs: /kaggle/working/rft_data_<RUN_NAME>.zip (training data) and
# /kaggle/working/rft_samples_<RUN_NAME>.zip (raw samples, for resume / debugging).
RUN_NAME = 'rft_p1b'

# TASK.md §1 Train Pool: public tasks that are not in dev60, not in
# eval/env_broken.json, minus rich_3882 (shares base_commit with a dev task).
# 57 tasks = fastapi 33 / rich 19 / requests 5.
TRAIN_IDS = [
    'fastapi_11355', 'fastapi_13537', 'fastapi_13786', 'fastapi_13920', 'fastapi_14077', 'fastapi_14099',
    'fastapi_14258', 'fastapi_14266', 'fastapi_14301', 'fastapi_14306', 'fastapi_14360', 'fastapi_14372',
    'fastapi_14419', 'fastapi_14459', 'fastapi_14463', 'fastapi_14485', 'fastapi_14492', 'fastapi_14512',
    'fastapi_14583', 'fastapi_14609', 'fastapi_14791', 'fastapi_14851', 'fastapi_14962', 'fastapi_14964',
    'fastapi_14986', 'fastapi_15030', 'fastapi_15588', 'fastapi_15661', 'fastapi_15800', 'fastapi_5077',
    'fastapi_9425', 'fastapi_9555', 'fastapi_9753', 'requests_6644', 'requests_7309', 'requests_7315',
    'requests_7433', 'requests_7505', 'rich_2943', 'rich_3006', 'rich_3052', 'rich_3061',
    'rich_3180', 'rich_3471', 'rich_3472', 'rich_3518', 'rich_3521', 'rich_3676',
    'rich_3772', 'rich_3905', 'rich_3930', 'rich_3935', 'rich_3942', 'rich_3953',
    'rich_4070', 'rich_4076', 'rich_4079',
]

# Paired Eval (dev60). Must never be sampled or trained on; checked below.
DEV60_IDS = [
    'fastapi_13713', 'rich_3063', 'requests_6629', 'httpx_3672', 'fastapi_11194', 'rich_3043',
    'fastapi_12942', 'rich_3067', 'fastapi_14246', 'fastapi_14262', 'rich_3105', 'fastapi_14349',
    'requests_6589', 'rich_2725', 'fastapi_5624', 'fastapi_14361', 'rich_3278', 'fastapi_14303',
    'rich_3468', 'fastapi_14297', 'rich_3470', 'fastapi_13207', 'requests_7205', 'fastapi_14430',
    'rich_3454', 'fastapi_14448', 'rich_3480', 'fastapi_14455', 'fastapi_14458', 'rich_3506',
    'fastapi_14482', 'rich_3535', 'requests_7328', 'fastapi_14487', 'rich_3675', 'fastapi_14371',
    'fastapi_14605', 'rich_3777', 'fastapi_14479', 'rich_3718', 'fastapi_14786', 'requests_7427',
    'fastapi_14794', 'rich_3934', 'fastapi_14616', 'rich_3894', 'fastapi_14873', 'rich_3938',
    'fastapi_14953', 'fastapi_14978', 'rich_3944', 'requests_7502', 'fastapi_15023', 'rich_4006',
    'fastapi_15280', 'fastapi_15589', 'rich_4075', 'fastapi_15745', 'rich_4077', 'fastapi_15785',
]

# None = the competition's tasks.jsonl / snapshots. Override to sample your own
# tasks (same schema; one snapshot per task in SNAPSHOTS_DIR_OVERRIDE).
TASKS_PATH_OVERRIDE = None
SNAPSHOTS_DIR_OVERRIDE = None

# Sampling (TASK.md §2.1). The proxy forces temperature/top_p on every request;
# thinking settings come from the submission unchanged (must match the final submission).
SAMPLE_TEMPERATURE = 0.7
SAMPLE_TOP_P = 0.95
MAX_TOOL_CALLS = 28
# Harness wall-clock budget while sampling: effectively none (TASK.md: no wall-clock
# timeout, so concurrency cannot truncate trajectories). The proxy rewrites the time
# allowance the model sees back to PROMPT_TIME_MINUTES, so prompts match the submission.
SAMPLING_MAX_TIME_MINUTES = 60
PROMPT_TIME_MINUTES = None  # None = max_time_minutes from the submission's eval_config.yaml

# Adaptive K: K_FIRST per task; tasks that went K_FIRST/K_FIRST, or 0/K_FIRST but
# edited the gold region (same_lines / same_region), get K_EXTRA more.
K_FIRST = 2
K_EXTRA = 2
# Parallel agent sessions against one vLLM server. TASK.md: 6-8; drop to 4-6 if
# CPU-bound (each sandbox takes ~2 vCPU).
CONCURRENCY = 6

# Selection (TASK.md §2.2-2.3, revised). The goal is fewer steps, so a task qualifies when
# it has a pass that is also efficient -- sometimes-solved or all-pass alike: an all-pass
# task whose passes vary in length is "sometimes efficient".
EDIT_CALL_LIMIT = 12          # the chosen trajectory must make its first source edit by this call
ALL_PASS_CALLS_LIMIT = 20     # all-pass tasks only: take their efficient mode, not just any pass
MAX_PER_TASK_ALL_PASS = 1     # trajectories per all-pass task (keeps easy tasks from dominating)
MAX_PER_TASK_SOMETIMES = 2    # trajectories per sometimes-solved task
# A pass only counts as a positive when its patch touches the reference files: a no_edit /
# scratch_only pass means the target tests pass without a source fix, a wrong_file pass
# needs a manual look.
GOOD_LOCS = ('same_lines', 'same_region', 'same_file')
# Key-turn loss mask: after the last kept source edit, train at most this many verification
# turns before submit_patch; later ones are masked as post_edit_excess.
POST_EDIT_KEEP = 2
MAX_SEQ_LEN = 16384           # training max_len; longer prefix chains are split into new segments

# Stop STARTING new samples after this many hours since notebook start. Kaggle kills
# sessions at 12 h; running samples, verification and packaging need the rest.
STOP_NEW_SAMPLES_AFTER_HOURS = 9.5

# Your submission: path to submission.zip or to a directory containing agent.yaml.
# None = auto-detect among the attached datasets.
SUBMISSION_INPUT = None

# Same test-environment fix as the dev-eval notebook (cell 2c). Keep it on: the
# pass/fail labels that drive the filtering come from Phase-2 verification.
FIX_TEST_ENV = True

# Resume: path to an attached previous version's output (a dir holding rft_samples_*/
# or rft_samples_*.zip). Finished samples are reused, only the missing ones run.
RESUME_DIR = None
# True = no sampling (no GPU needed): rebuild the training data from RESUME_DIR's samples.
SKIP_SAMPLING = False

assert len(TRAIN_IDS) == len(set(TRAIN_IDS)), 'duplicate ids in TRAIN_IDS'
assert not set(TRAIN_IDS) & set(DEV60_IDS), f'Paired-Eval tasks in TRAIN_IDS: {sorted(set(TRAIN_IDS) & set(DEV60_IDS))}'
assert not (SKIP_SAMPLING and not RESUME_DIR), 'SKIP_SAMPLING needs RESUME_DIR (samples to rebuild from)'
print(f'{RUN_NAME}: {len(TRAIN_IDS)} train tasks, K={K_FIRST}+{K_EXTRA}, concurrency={CONCURRENCY}, '
      f'temperature={SAMPLE_TEMPERATURE}, skip_sampling={SKIP_SAMPLING}')

## 1. Environment Configuration and Package Installation

与 dev-eval notebook 的 cell 1 相同。

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Your Submission

复制 submission 后只改 `eval_config.yaml` 的采样预算：`max_tool_calls=28`，墙钟时间放宽。指纹按改动之前的原 submission 计算。

In [ ]:
import hashlib
import json
import zipfile
from collections import Counter

import yaml
from adk_submission import validate_directory
from swegemma.config import build_submission_limits
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)
TASKS_PATH = Path(TASKS_PATH_OVERRIDE) if TASKS_PATH_OVERRIDE else DATA_DIR / 'tasks.jsonl'
SNAPSHOTS_DIR = Path(SNAPSHOTS_DIR_OVERRIDE) if SNAPSHOTS_DIR_OVERRIDE else DATA_DIR / 'snapshots'
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')
AGENT_DIR = WORKING_DIR / 'submission_src'
SAMPLES_DIR = WORKING_DIR / f'rft_samples_{RUN_NAME}'   # <task>/k<k>/{meta.json, requests.jsonl, ...}
DATA_OUT = WORKING_DIR / f'rft_data_{RUN_NAME}'
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)


def _zip_has_root_agent(z: Path) -> bool:
    try:
        with zipfile.ZipFile(z) as zf:
            return 'agent.yaml' in zf.namelist()
    except zipfile.BadZipFile:
        return False


def _in_resume(p: Path) -> bool:
    """Inside the attached previous output (RESUME_DIR), which also holds a submission_src/ copy."""
    return bool(RESUME_DIR) and Path(RESUME_DIR).resolve() in (p.resolve(), *p.resolve().parents)


def find_submission() -> Path:
    """Locate the attached submission: a zip with agent.yaml at its root, or a
    directory holding agent.yaml (Kaggle may auto-extract uploaded zips)."""
    if SUBMISSION_INPUT:
        roots = [Path(SUBMISSION_INPUT)]
    else:
        inp = Path('/kaggle/input')
        roots = [p for p in (inp / 'datasets').glob('*/*') if p.is_dir()]
        roots += [p for p in inp.iterdir() if p.is_dir() and p.name not in ('competitions', 'models', 'datasets')]
        roots = [r for r in roots if r.resolve() != WHEELHOUSE_DIR.resolve()]
    zips, dirs, bad = [], [], []
    for root in roots:
        if root.is_file():
            (zips if _zip_has_root_agent(root) else bad).append(root)
            continue
        for z in root.rglob('*.zip'):
            if not _in_resume(z):
                (zips if _zip_has_root_agent(z) else bad).append(z)
        dirs += [p.parent for p in root.rglob('agent.yaml') if not _in_resume(p)]
    if bad:
        print('Ignoring zips without agent.yaml at the archive root (the real submission would be rejected):', bad)
    candidates = zips or dirs
    if len(candidates) != 1:
        raise RuntimeError(
            f'Expected exactly one submission, found zips={zips} dirs={dirs}. '
            'Attach your submission dataset, or set SUBMISSION_INPUT in the run config.'
        )
    return candidates[0]


def submission_fingerprint(root: Path) -> str:
    """sha256 over (relative path, content) of every file. Must match scripts/package_submission.py."""
    h = hashlib.sha256()
    files = sorted((p for p in root.rglob('*') if p.is_file()), key=lambda p: p.relative_to(root).as_posix())
    for p in files:
        h.update(p.relative_to(root).as_posix().encode())
        h.update(b'\0')
        h.update(p.read_bytes())
        h.update(b'\0')
    return h.hexdigest()[:12]


SUBMISSION_SRC = find_submission()
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
if SUBMISSION_SRC.suffix == '.zip':
    with zipfile.ZipFile(SUBMISSION_SRC) as zf:
        zf.extractall(AGENT_DIR)
else:
    shutil.copytree(SUBMISSION_SRC, AGENT_DIR)

# Same structural checks the real submission goes through -- fail here, not after vLLM startup
for p in AGENT_DIR.rglob('*'):
    assert not p.is_symlink(), f'Symlinks are not permitted: {p.relative_to(AGENT_DIR)}'
limits, gen_constraints = build_submission_limits()
validate_directory(AGENT_DIR, limits=limits)
SUBMISSION_FINGERPRINT = submission_fingerprint(AGENT_DIR)  # before the sampling overrides below

# Sampling budgets (TASK.md §2.1): 28 tool calls, no wall-clock limit. The time the
# model is told it has stays the submission's value (rewritten by the proxy, section 4).
eval_config_file = next(
    (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := AGENT_DIR / name).is_file()),
    AGENT_DIR / 'eval_config.yaml',
)
raw_eval_cfg = (yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {}) if eval_config_file.is_file() else {}
eval_section = dict(raw_eval_cfg.get('evaluation', raw_eval_cfg))
SUBMISSION_EVAL_SECTION = dict(eval_section)
if PROMPT_TIME_MINUTES is None:
    PROMPT_TIME_MINUTES = float(eval_section.get('max_time_minutes', 60.0))
assert float(PROMPT_TIME_MINUTES) != float(SAMPLING_MAX_TIME_MINUTES), (
    'SAMPLING_MAX_TIME_MINUTES must differ from PROMPT_TIME_MINUTES so the proxy can tell the values apart'
)
eval_section.update(max_tool_calls=MAX_TOOL_CALLS, max_time_minutes=SAMPLING_MAX_TIME_MINUTES)
eval_config_file.write_text(yaml.safe_dump({'evaluation': eval_section}, sort_keys=False), encoding='utf-8')

_sampling_path = AGENT_DIR / 'configs' / 'sampling.yaml'
SUBMISSION_SAMPLING = yaml.safe_load(_sampling_path.read_text(encoding='utf-8')) if _sampling_path.is_file() else None

tasks = load_tasks(TASKS_PATH)
TASK_BY_ID = {t.instance_id: t for t in tasks}
# raw rows: the reference `patch` drives gold-file / gold-region detection
RAW_TASKS = {d['instance_id']: d for d in map(json.loads, filter(str.strip, TASKS_PATH.read_text(encoding='utf-8').splitlines()))}
_missing = [i for i in TRAIN_IDS if i not in TASK_BY_ID]
assert not _missing, f'TRAIN_IDS not found in {TASKS_PATH}: {_missing}'

print(f'Submission source: {SUBMISSION_SRC}')
print(f'Submission fingerprint: {SUBMISSION_FINGERPRINT}  <- must match scripts/package_submission.py')
for p in sorted(AGENT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(AGENT_DIR).as_posix()}')
print(f'--- submission eval_config: {SUBMISSION_EVAL_SECTION}')
print(f'--- sampling eval_config:   {eval_section}  (model is told {PROMPT_TIME_MINUTES} min)')
print(f'--- configs/sampling.yaml:  {SUBMISSION_SAMPLING}  (temperature/top_p overridden to {SAMPLE_TEMPERATURE}/{SAMPLE_TOP_P})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH}; train pool by repo:',
      dict(Counter(RAW_TASKS[i]['repo'] for i in TRAIN_IDS)))

### 2c. Test-Environment Fix (missing test deps + src-layout)

与 dev-eval notebook 的 cell 2c 相同。Phase-2 的通过/失败标签决定哪些题是摇摆任务。

In [ ]:
# In sandbox='subprocess' mode the harness never installs anything from
# EvalConfig.wheels_dir: tests run in a per-task venv that sees THIS kernel's
# site-packages, with PYTHONPATH=<sandbox>/workspace. That caused most of the
# env_error bucket in the 1016/1017 runs. So:
#  1. test deps missing from the Kaggle image (inline_snapshot, dirty_equals,
#     pytest-httpbin, trustme) are pip-installed into the kernel from the
#     tianyuxu2/gemma-4-extra-test-wheels dataset (already-satisfied deps are kept);
#  2. src-layout repos (psf/requests) would import the kernel's installed
#     `requests`, not the patched workspace copy; a .pth hook puts
#     <workspace>/src first on sys.path, but only inside swegemma sandboxes.
# With this fix the reference patches pass on 114/129 tasks (74/129 without).
import site
import tempfile
import venv

ENV_FIX = {'enabled': FIX_TEST_ENV}
if FIX_TEST_ENV:
    _roots = [p for p in (Path('/kaggle/input') / 'datasets').glob('*/*') if p.is_dir()]
    _roots += [p for p in Path('/kaggle/input').iterdir() if p.is_dir() and p.name not in ('competitions', 'models', 'datasets')]
    _hits = [w for r in _roots for w in r.rglob('inline_snapshot-*.whl')]
    if not _hits:
        raise RuntimeError(
            'FIX_TEST_ENV=True but no inline_snapshot wheel found: attach the '
            'tianyuxu2/gemma-4-extra-test-wheels dataset (or set FIX_TEST_ENV = False).'
        )
    EXTRA_WHEELS_DIR = _hits[0].parent

    pip_freeze = lambda: set(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout.split())
    _before = pip_freeze()
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', str(EXTRA_WHEELS_DIR),
         'inline-snapshot', 'dirty-equals', 'pytest-httpbin', 'trustme'],
        check=True,
    )
    _after = pip_freeze()
    ENV_FIX.update(wheels_dir=str(EXTRA_WHEELS_DIR), installed=sorted(_after - _before), replaced=sorted(_before - _after))
    print('extra test deps installed:', ENV_FIX['installed'])
    print('replaced:', ENV_FIX['replaced'])

    SRC_LAYOUT_HOOK = (
        "import os, sys; _s = [os.path.join(p, 'src') for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) "
        "if 'swegemma_sandbox_' in p and os.path.isdir(os.path.join(p, 'src'))]; "
        "sys.path[:0] = [p for p in _s if p not in sys.path]\n"
    )
    for sp in site.getsitepackages():
        if Path(sp).is_dir():
            (Path(sp) / '00_swegemma_src_layout.pth').write_text(SRC_LAYOUT_HOOK)
            ENV_FIX['src_layout_hook'] = sp
            print('src-layout hook ->', sp)
            break

    # self-check in a system-site venv like the harness's: deps importable, src/ first
    _chk = Path(tempfile.mkdtemp(prefix='swegemma_sandbox_selfcheck_'))
    (_chk / 'workspace' / 'src').mkdir(parents=True)
    venv.create(_chk / 'venv', system_site_packages=True, with_pip=False, symlinks=True)
    _res = subprocess.run(
        [str(_chk / 'venv' / 'bin' / 'python3'), '-c',
         'import sys, inline_snapshot, dirty_equals, pytest_httpbin, trustme; print(sys.path)'],
        env={**os.environ, 'PYTHONPATH': str(_chk / 'workspace')}, capture_output=True, text=True,
    )
    if _res.returncode != 0 or str(_chk / 'workspace' / 'src') not in _res.stdout:
        raise RuntimeError(f'sandbox self-check FAILED\n{_res.stdout}\n{_res.stderr[-2000:]}')
    ENV_FIX['self_check'] = 'OK'
    print('sandbox self-check: OK')
else:
    print('FIX_TEST_ENV=False: running with the stock test environment (expect env_error failures).')

## 3. Helpers

内联了 `eval/refdiff.py`（gold 文件和定位判断）、`eval/trace_stats.py`（调用计数和源码编辑判断），以及轨迹解析和 prefix 链打包。这些都是纯函数，不依赖 GPU。

In [ ]:
# Pure functions, no GPU: section 6/7 can be re-run on their own.
import bisect
import copy
import os
import re
from collections import Counter
from dataclasses import dataclass, field
from typing import Iterator

# ---- inlined from eval/refdiff.py: agent patch vs reference patch ----
_HUNK_RE = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@ ?(.*)$")
_DEF_RE = re.compile(r"^(\s*)(?:async\s+)?def\s+(\w+)\s*\(")
_CLASS_RE = re.compile(r"^(\s*)class\s+(\w+)\b")

# two edits within this many lines of each other count as "same region"
NEAR_LINES = 30

# same rule the harness uses to strip test files from agent patches
PROTECTED_TEST_RE = re.compile(r"(^|/)(tests?|testing)/.*\.py$|(^|/)test_[^/]*\.py$|_test\.py$|(^|/)conftest\.py$")


def _strip_prefix(p: str) -> str:
    p = p.strip().split("\t")[0]
    if p.startswith(("a/", "b/")):
        p = p[2:]
    return p


@dataclass
class Hunk:
    path: str
    old_start: int
    header: str          # git funcname context, raw (keeps indentation)
    lines: list[str]     # body lines, each starting with ' ', '+' or '-'
    created: bool = False  # file did not exist before (--- /dev/null)


def iter_hunks(diff: str) -> Iterator[Hunk]:
    """Yield hunks of a unified diff. Hunk bodies are consumed by the line
    counts in the @@ header, so plain multi-file diffs (no `diff --git`
    separators, like the reference patches) split correctly."""
    lines = (diff or "").splitlines()
    i, n = 0, len(lines)
    old_path = new_path = None
    while i < n:
        line = lines[i]
        if line.startswith("--- "):
            old_path = _strip_prefix(line[4:])
            i += 1
            continue
        if line.startswith("+++ "):
            new_path = _strip_prefix(line[4:])
            i += 1
            continue
        m = _HUNK_RE.match(line)
        if not m:
            i += 1
            continue
        path = old_path if new_path in (None, "/dev/null") else new_path
        old_left = int(m.group(2)) if m.group(2) is not None else 1
        new_left = int(m.group(4)) if m.group(4) is not None else 1
        body: list[str] = []
        i += 1
        while i < n and (old_left > 0 or new_left > 0):
            b = lines[i]
            if b.startswith("\\"):           # "\ No newline at end of file"
                i += 1
                continue
            tag = b[:1] or " "               # some tools drop the space on blank context lines
            if tag == "-":
                old_left -= 1
            elif tag == "+":
                new_left -= 1
            elif tag == " ":
                old_left -= 1
                new_left -= 1
            else:
                break
            body.append(b if b else " ")
            i += 1
        yield Hunk(path=path or "?", old_start=int(m.group(1)), header=m.group(5), lines=body,
                   created=old_path == "/dev/null")


@dataclass
class FileEdit:
    path: str
    # line numbers in the ORIGINAL file that were removed, or next to which
    # lines were inserted
    old_lines: set[int] = field(default_factory=set)
    created: bool = False


def parse_diff(diff: str) -> dict[str, FileEdit]:
    files: dict[str, FileEdit] = {}
    for h in iter_hunks(diff):
        fe = files.setdefault(h.path, FileEdit(h.path))
        fe.created = fe.created or h.created
        old_no = h.old_start
        for b in h.lines:
            if b[0] == "-":
                fe.old_lines.add(old_no)
                old_no += 1
            elif b[0] == "+":
                fe.old_lines.update((max(1, old_no - 1), old_no))
            else:
                old_no += 1
    return files


def source_edits(edits: dict[str, FileEdit]) -> dict[str, FileEdit]:
    """Drop test/conftest files -- the harness strips them from agent patches anyway."""
    return {p: e for p, e in edits.items() if not PROTECTED_TEST_RE.search(p)}


@dataclass
class Localization:
    ref_files: list[str]
    agent_files: list[str]
    file_hit: bool          # agent touched >=1 reference source file
    region_hit: bool        # some agent edit within NEAR_LINES of a reference edit
    overlap_hit: bool       # some agent edit within 3 lines of a reference edit
    missed_files: list[str]  # reference files the agent never touched
    extra_files: list[str]   # files the agent touched that the reference did not
    scratch_files: list[str]  # files the agent created that the reference did not (repro.py ...)

    @property
    def real_files(self) -> list[str]:
        """Agent-edited source files, minus scratch scripts."""
        return [f for f in self.agent_files if f not in self.scratch_files]

    @property
    def label(self) -> str:
        if not self.agent_files:
            return "no_edit"
        if not self.real_files:
            return "scratch_only"
        if self.overlap_hit:
            return "same_lines"
        if self.region_hit:
            return "same_region"
        if self.file_hit:
            return "same_file"
        return "wrong_file"


def _min_dist(a: set[int], b: set[int]) -> int:
    if not a or not b:
        return 10**9
    sb = sorted(b)
    best = 10**9
    for x in a:
        i = bisect.bisect_left(sb, x)
        for j in (i - 1, i):
            if 0 <= j < len(sb):
                best = min(best, abs(sb[j] - x))
    return best


def localize(ref_patch: str, agent_patch: str) -> Localization:
    ref = source_edits(parse_diff(ref_patch))
    agt = source_edits(parse_diff(agent_patch))
    common = set(ref) & set(agt)
    best = min((_min_dist(ref[p].old_lines, agt[p].old_lines) for p in common), default=10**9)
    return Localization(
        ref_files=sorted(ref),
        agent_files=sorted(agt),
        file_hit=bool(common),
        region_hit=best <= NEAR_LINES,
        overlap_hit=best <= 3,
        missed_files=sorted(set(ref) - set(agt)),
        extra_files=sorted(set(agt) - set(ref)),
        scratch_files=sorted(p for p, e in agt.items() if e.created and p not in ref),
    )


# ---- inlined from eval/trace_stats.py: what counts as a tool call / a source edit ----
EDIT_TOOLS = {"edit_file", "write_file"}
SCRATCH_HINTS = ("repro", "reproduce", "check_", "verify_", "debug", "scratch")
FREE_TOOLS = {"submit_patch", "get_status"}  # do not count toward the tool-call budget


def _is_source_edit(fn: str, args: dict) -> bool:
    if fn in EDIT_TOOLS:
        path = os.path.basename(args.get("filepath") or "")
        return not any(h in path for h in SCRATCH_HINTS)
    if fn == "run_command":
        cmd = args.get("command") or ""
        return "sed -i" in cmd or (".write_text(" in cmd) or ("open(" in cmd and "'w'" in cmd)
    return False


# ---- trajectories from the logging proxy (samples/<task>/k<k>/requests.jsonl) ----
def gold_files(task_id: str) -> list[str]:
    """Source files the reference patch edits (test files excluded)."""
    return sorted(source_edits(parse_diff(RAW_TASKS[task_id]['patch'])))


def _norm_path(p) -> str:
    p = str(p or '').strip().strip('\'"')
    for prefix in ('/workspace/', './'):
        if p.startswith(prefix):
            p = p[len(prefix):]
    return p.lstrip('/')


def _mentions_gold(fn: str, args: dict, gold: list[str]) -> bool:
    """The call reads (or edits) a reference-patch file: read_file/edit on it, or a
    shell command that names its path (sed -n, cat, grep -n ... path)."""
    if fn in ('read_file', 'edit_file', 'write_file'):
        return _norm_path(args.get('filepath') or args.get('path')) in gold
    if fn == 'run_command':
        cmd = args.get('command') or ''
        return any(g in cmd for g in gold)
    return False


def parse_tool_calls(message: dict) -> list[tuple[str, dict]]:
    out = []
    for tc in message.get('tool_calls') or []:
        fn = tc.get('function') or {}
        args = fn.get('arguments')
        if isinstance(args, str):
            try:
                args = json.loads(args) if args.strip() else {}
            except json.JSONDecodeError:
                args = {'_raw': args}
        out.append((fn.get('name') or '?', args if isinstance(args, dict) else {'_raw': args}))
    return out


def parse_sse(text: str) -> dict:
    """Rebuild a non-streaming chat response (incl. vLLM token ids) from SSE chunks."""
    prompt_ids, token_ids, content, reasoning, calls, finish = None, [], [], [], {}, None
    for line in text.splitlines():
        if not line.startswith('data:'):
            continue
        data = line[5:].strip()
        if data == '[DONE]':
            break
        try:
            chunk = json.loads(data)
        except json.JSONDecodeError:
            continue
        if chunk.get('prompt_token_ids') is not None:
            prompt_ids = chunk['prompt_token_ids']
        for c in chunk.get('choices') or []:
            token_ids += c.get('token_ids') or []
            d = c.get('delta') or {}
            content.append(d.get('content') or '')
            reasoning.append(d.get('reasoning_content') or d.get('reasoning') or '')
            for tc in d.get('tool_calls') or []:
                slot = calls.setdefault(tc.get('index', 0), {'function': {'name': '', 'arguments': ''}})
                f = tc.get('function') or {}
                slot['function']['name'] += f.get('name') or ''
                slot['function']['arguments'] += f.get('arguments') or ''
            finish = c.get('finish_reason') or finish
    message = {'content': ''.join(content), 'reasoning_content': ''.join(reasoning),
               'tool_calls': [calls[i] for i in sorted(calls)]}
    return {'prompt_token_ids': prompt_ids, 'choices': [{'message': message, 'token_ids': token_ids, 'finish_reason': finish}]}


def load_turns(path: Path, with_ids: bool = True) -> list[dict]:
    """One dict per successful chat request of a sample, in send order.

    `superseded`: the same prompt was sent again right after (a retry), so this
    answer was thrown away and its tool calls never ran. with_ids=False drops the
    token ids and the request to keep memory low when scanning every sample.
    """
    recs = [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines() if l.strip()] if path.exists() else []
    recs.sort(key=lambda r: r.get('ts_start', 0))
    turns = []
    for r in recs:
        if r.get('status') != 200:
            continue
        resp = parse_sse(r.get('response_text') or '') if r.get('stream') else (r.get('response') or {})
        ch = (resp.get('choices') or [{}])[0]
        msg = ch.get('message') or {}
        p, c = resp.get('prompt_token_ids'), ch.get('token_ids')
        turns.append({
            'prompt_ids': p, 'completion_ids': c,
            'prompt_key': hash(tuple(p)) if p else None,
            'has_ids': bool(p) and c is not None,
            'n_prompt': len(p or []), 'n_completion': len(c or []),
            'finish_reason': ch.get('finish_reason'),
            'tool_calls': parse_tool_calls(msg),
            'content': msg.get('content') or '',
            'reasoning': msg.get('reasoning_content') or msg.get('reasoning') or '',
            'rewrites': r.get('rewrites'),
            'request': r.get('request'),
        })
    for i, t in enumerate(turns):
        t['idx'] = i
        t['superseded'] = i + 1 < len(turns) and t['prompt_key'] is not None and t['prompt_key'] == turns[i + 1]['prompt_key']
        if not with_ids:
            t['prompt_ids'] = t['completion_ids'] = t['request'] = None
    return turns


def analyze_turns(turns: list[dict], gold: list[str]) -> dict:
    """Budget usage (counted like eval/trace_stats.py) and the turns that bound the loss window."""
    calls = 0
    first_edit_call = first_edit_turn = gold_read_turn = None
    submit_turns = []
    for t in turns:
        if t['superseded']:
            continue
        for fn, args in t['tool_calls']:
            if fn not in FREE_TOOLS:
                calls += 1
            if first_edit_call is None and _is_source_edit(fn, args):
                first_edit_call, first_edit_turn = calls, t['idx']
            if gold_read_turn is None and _mentions_gold(fn, args, gold):
                gold_read_turn = t['idx']
            if fn == 'submit_patch':
                submit_turns.append(t['idx'])
    live = [t for t in turns if not t['superseded']]
    return {
        'calls': calls,
        'first_edit_call': first_edit_call,
        'first_edit_turn': first_edit_turn,
        'gold_read_turn': gold_read_turn,
        'submit_turn': submit_turns[-1] if submit_turns else None,
        'n_turns': len(live),
        'n_retries': len(turns) - len(live),
        'has_ids': bool(live) and all(t['has_ids'] for t in live),
        'completion_tokens': sum(t['n_completion'] for t in live),
        'max_prompt_tokens': max((t['n_prompt'] for t in live), default=0),
    }


def loss_window(turns: list[dict], a: dict) -> tuple[int, int] | None:
    """TASK.md §2.3: assistant turns after the first read of a gold file, up to and
    including the submit_patch turn. The first source edit is always inside the
    window (it may happen before any explicit read of the gold file)."""
    starts = []
    if a['gold_read_turn'] is not None:
        starts.append(a['gold_read_turn'] + 1)
    if a['first_edit_turn'] is not None:
        starts.append(a['first_edit_turn'])
    if not starts or not turns:
        return None
    start = min(starts)
    end = a['submit_turn'] if a['submit_turn'] is not None else turns[-1]['idx']
    return (start, end) if start <= end else None


def _strip_end(ids: list[int], end_ids: set[int]) -> list[int]:
    j = len(ids)
    while j > 0 and ids[j - 1] in end_ids:
        j -= 1
    return ids[:j]


def pack_turns(turns: list[dict], selected: set[int], end_ids: set[int], max_len: int):
    """Pack one trajectory into training sequences (TASK.md §2.3 "prefix chains").

    Turn t joins the current sequence when its prompt starts with the previous
    turn's prompt + generated tokens (end-of-turn tokens excepted); otherwise
    (context compaction, re-rendered history, or > max_len) a new sequence starts
    at t's full prompt. Every sequence is therefore exactly what the model saw.
    labels = token id on the generated tokens of selected turns (+ the end-of-turn
    tokens right after them), -100 everywhere else. Sequences without a selected
    turn are dropped.
    """
    segs, stats = [], Counter()
    cur = None

    def close():
        if cur and any(sel for _, _, _, sel in cur['spans']):
            ids = cur['ids']
            labels = [-100] * len(ids)
            for _, s, e, sel in cur['spans']:
                if not sel:
                    continue
                while e < len(ids) and ids[e] in end_ids:
                    e += 1
                labels[s:e] = ids[s:e]
            segs.append({'input_ids': ids, 'labels': labels,
                         'turns': [i for i, _, _, _ in cur['spans']],
                         'loss_turns': [i for i, _, _, sel in cur['spans'] if sel]})

    for t in turns:
        if t['superseded'] or not t['has_ids']:
            if t['idx'] in selected:
                stats['selected_turn_without_ids'] += 1
            continue
        p, c = t['prompt_ids'], t['completion_ids']
        gen = _strip_end(c, end_ids)
        full = p + c
        sel = t['idx'] in selected
        if len(full) > max_len:
            stats['turn_too_long_selected' if sel else 'turn_too_long_context'] += 1
            close()
            cur = None
            continue
        if cur is not None:
            k = cur['p_len'] + cur['g_len']
            if len(p) > k and p[:k] == cur['ids'][:k]:
                stats['merged'] += 1
            else:
                stats['chain_break'] += 1
                close()
                cur = None
        if cur is None:
            cur = {'spans': []}
        cur['ids'] = full
        cur['p_len'], cur['g_len'] = len(p), len(gen)
        cur['spans'].append((t['idx'], len(p), len(p) + len(gen), sel))
    close()
    return segs, stats


# ---- key-turn loss mask (TASK.md §2.3, revised) ----
FILE_TOOLS = {'edit_file', 'write_file', 'read_file'}


def _parse_result(content) -> dict | None:
    if isinstance(content, list):  # content parts
        content = ''.join(p.get('text', '') for p in content if isinstance(p, dict))
    if not isinstance(content, str):
        return None
    try:
        out = json.loads(content)
    except json.JSONDecodeError:
        return {'_raw': content}
    return out if isinstance(out, dict) else {'_raw': out}


def attach_results(turns: list[dict]) -> int:
    """Set t['results'] to the parsed tool results of t's calls, read from the next live
    turn's request (the tool messages after its last assistant message). None when they
    cannot be matched. Needs load_turns(with_ids=True). Returns how many turns with calls
    could not be matched although a next turn exists."""
    live = [t for t in turns if not t['superseded']]
    unmatched = 0
    for t in turns:
        t['results'] = None
    for i, t in enumerate(live):
        if not t['tool_calls'] or i + 1 == len(live):
            continue
        msgs = (live[i + 1].get('request') or {}).get('messages') or []
        last = max((j for j, m in enumerate(msgs) if m.get('role') == 'assistant'), default=None)
        tool_msgs = [m for m in msgs[last + 1:] if m.get('role') == 'tool'] if last is not None else []
        if len(tool_msgs) != len(t['tool_calls']):
            unmatched += 1
            continue
        by_id = {m.get('tool_call_id'): m for m in tool_msgs}
        ids = [tc.get('id') for tc in msgs[last].get('tool_calls') or []]
        if len(ids) == len(tool_msgs) and all(x in by_id for x in ids):
            tool_msgs = [by_id[x] for x in ids]
        t['results'] = [_parse_result(m.get('content')) for m in tool_msgs]
    return unmatched


def _call_ok(fn: str, res) -> bool:
    """False when the call itself failed: a file tool returned an error, or the harness
    rejected the arguments. run_command errors do not count (failing tests, empty grep).
    Unknown results count as ok."""
    if not isinstance(res, dict):
        return True
    if 'error' in res and 'status' not in res:  # "Invoking `x()` failed ..." (bad arguments)
        return False
    return not (fn in FILE_TOOLS and res.get('status') == 'error')


def _call_key(fn: str, args: dict) -> str:
    return fn + json.dumps(args, sort_keys=True, ensure_ascii=False)


def patch_files(tid: str, k: int) -> set[str]:
    """Source files of the sample's submitted patch (test files excluded)."""
    p = sample_dir(tid, k) / 'patch.diff'
    return set(source_edits(parse_diff(p.read_text(encoding='utf-8')))) if p.exists() else set()


def key_turns(turns: list[dict], a: dict, final_files: set[str]) -> tuple[set[int], dict[int, str]]:
    """Loss turns: the loss_window() turns minus those not worth copying. Returns
    (selected turn idxs, {masked turn idx: reason}); the first matching reason wins:
      failed_tool       a file tool errored, or the harness rejected the arguments
      status_only       only get_status
      discarded_edit    edit_file on a source file that is not in the submitted patch
      duplicate         every call repeats an earlier identical call, no source edit since
      post_edit_excess  after the last kept source edit, verification turns past POST_EDIT_KEEP
    The first source-edit turn and the submit turn are exempt from the last two. Masked
    turns stay in the context of later turns: the model did see them."""
    win = loss_window(turns, a)
    if win is None:
        return set(), {}
    start, end = win
    final = {_norm_path(f) for f in final_files}
    exempt = {a['first_edit_turn'], a['submit_turn']}
    reasons, seen, n_edits = {}, {}, 0
    cand, kept_edit_turns = [], []
    for t in turns:
        if t['superseded']:
            continue
        calls = list(zip(t['tool_calls'], t.get('results') or [None] * len(t['tool_calls'])))
        real = [(fn, args) for (fn, args), _ in calls if fn not in FREE_TOOLS]
        dup = bool(real) and all(seen.get(_call_key(fn, args)) == n_edits for fn, args in real)
        discarded = False
        for (fn, args), res in calls:
            if not (_is_source_edit(fn, args) and _call_ok(fn, res)):
                continue
            n_edits += 1
            path = _norm_path(args.get('filepath') or args.get('path'))
            if fn == 'edit_file' and final and path not in final and not PROTECTED_TEST_RE.search(path):
                discarded = True
            elif fn == 'run_command' or path in final:
                kept_edit_turns.append(t['idx'])
        for fn, args in real:
            seen[_call_key(fn, args)] = n_edits
        if not (start <= t['idx'] <= end and t['has_ids'] and t['finish_reason'] != 'length'):
            continue
        cand.append(t['idx'])
        if any(not _call_ok(fn, res) for (fn, _), res in calls):
            reasons[t['idx']] = 'failed_tool'
        elif t['tool_calls'] and all(fn == 'get_status' for fn, _ in t['tool_calls']):
            reasons[t['idx']] = 'status_only'
        elif discarded:
            reasons[t['idx']] = 'discarded_edit'
        elif dup and t['idx'] not in exempt:
            reasons[t['idx']] = 'duplicate'
    if kept_edit_turns:
        last_edit, kept = max(kept_edit_turns), 0
        for i in cand:
            if i <= last_edit or i in exempt or i in reasons:
                continue
            if kept < POST_EDIT_KEEP:
                kept += 1
            else:
                reasons[i] = 'post_edit_excess'
    return set(cand) - set(reasons), reasons


def render_prompt_ids(tokenizer, request: dict) -> list[int]:
    """Prompt ids from tokenizer.apply_chat_template, mirroring vLLM's preprocessing
    (tool-call arguments decoded from JSON strings)."""
    msgs = copy.deepcopy(request['messages'])
    for m in msgs:
        for tc in m.get('tool_calls') or []:
            f = tc.get('function') or {}
            if isinstance(f.get('arguments'), str):
                try:
                    f['arguments'] = json.loads(f['arguments'])
                except json.JSONDecodeError:
                    pass
    out = tokenizer.apply_chat_template(
        msgs, tools=request.get('tools'), add_generation_prompt=True, tokenize=True,
        **(request.get('chat_template_kwargs') or {}),
    )
    if hasattr(out, 'keys'):
        out = out['input_ids']
    return list(out)


print('helpers ready')

## 4. vLLM + Logging Proxy

每个并发 slot 有一个代理端口，并配一份指向它的 model registry。代理的作用：
- 记录原始请求和响应
- 向 vLLM 要 `prompt_token_ids` / `token_ids`
- 强制采样温度
- 把模型看到的时间预算改回 submission 的值

冒烟测试失败时会直接报错。

In [ ]:
import http.client
import http.server
import inspect
import threading

import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
if adapter_map:
    print(f'WARNING: the submission ships LoRA adapters {sorted(adapter_map)}: samples come from base+LoRA, '
          'not the base policy TASK.md §2.1 asks for.')
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

# ---------------------------------------------------------------------------
# Logging proxy (TASK.md §2.1). One proxy port per concurrent slot, so every
# request is attributed to the sample that slot is running. For each
# /v1/chat/completions request it:
#   1. rewrites the time budget the model is told ("Time allowance: X minutes" in
#      the task prompt, "max_time_minutes" in get_status output) from
#      SAMPLING_MAX_TIME_MINUTES back to PROMPT_TIME_MINUTES;
#   2. forces temperature / top_p to the sampling values;
#   3. asks vLLM for the exact prompt / generated token ids (return_token_ids);
#   4. logs request + full response, and hands the response to the harness
#      without the token-id fields.
# ---------------------------------------------------------------------------
VLLM_PORT = 8000
PROXY_BASE_PORT = 8100
UNMATCHED_DIR = SAMPLES_DIR / '_unmatched'
UNMATCHED_DIR.mkdir(parents=True, exist_ok=True)
_TIME_RES = (
    re.compile(r'(Time allowance: )(\d+(?:\.\d+)?)( minutes)'),
    re.compile(r'("max_time_minutes":\s*)(\d+(?:\.\d+)?)'),
)
_HOP_HEADERS = {'host', 'content-length', 'connection', 'keep-alive', 'transfer-encoding', 'accept-encoding',
                'proxy-connection', 'te', 'trailer', 'upgrade'}


def _rewrite_text(s: str) -> tuple[str, int]:
    n = 0

    def sub(m):
        nonlocal n
        if float(m.group(2)) != float(SAMPLING_MAX_TIME_MINUTES):
            return m.group(0)
        n += 1
        return m.group(1) + str(float(PROMPT_TIME_MINUTES)) + m.group(0)[m.end(2) - m.start():]

    for rx in _TIME_RES:
        s = rx.sub(sub, s)
    return s, n


def rewrite_request(body: dict) -> int:
    n = 0
    for msg in body.get('messages') or []:
        c = msg.get('content')
        if isinstance(c, str):
            msg['content'], k = _rewrite_text(c)
            n += k
        elif isinstance(c, list):
            for part in c:
                if isinstance(part, dict) and isinstance(part.get('text'), str):
                    part['text'], k = _rewrite_text(part['text'])
                    n += k
    body['temperature'] = SAMPLE_TEMPERATURE
    body['top_p'] = SAMPLE_TOP_P
    body['return_token_ids'] = True
    return n


class _ProxyHandler(http.server.BaseHTTPRequestHandler):
    protocol_version = 'HTTP/1.1'

    def log_message(self, *args):
        pass

    def do_GET(self):
        self._relay()

    def do_POST(self):
        self._relay()

    def do_DELETE(self):
        self._relay()

    def _relay(self):
        n = int(self.headers.get('Content-Length') or 0)
        raw = self.rfile.read(n) if n else b''
        body = rewrites = None
        if self.command == 'POST' and self.path.split('?')[0].rstrip('/').endswith('/chat/completions'):
            try:
                body = json.loads(raw)
                rewrites = rewrite_request(body)
                raw = json.dumps(body).encode()
            except Exception:  # not JSON: forward untouched, do not log
                body = None
        headers = {k: v for k, v in self.headers.items() if k.lower() not in _HOP_HEADERS}
        headers['Content-Length'] = str(len(raw))
        t0 = time.time()
        try:
            conn = http.client.HTTPConnection('127.0.0.1', VLLM_PORT, timeout=3600)
            conn.request(self.command, self.path, body=raw or None, headers=headers)
            resp = conn.getresponse()
            data, status = resp.read(), resp.status
            resp_headers = [(k, v) for k, v in resp.getheaders() if k.lower() not in _HOP_HEADERS]
            conn.close()
        except Exception as e:
            status, resp_headers = 502, [('Content-Type', 'application/json')]
            data = json.dumps({'error': {'message': f'logging proxy: {type(e).__name__}: {e}'}}).encode()
        out = data
        if body is not None:
            rec = {'ts_start': t0, 'ts_end': time.time(), 'status': status, 'rewrites': rewrites,
                   'stream': bool(body.get('stream')), 'request': body}
            try:
                if rec['stream']:  # extra fields in SSE chunks are ignored by the client
                    rec['response_text'] = data.decode('utf-8', 'replace')
                else:
                    resp_json = json.loads(data)
                    rec['response'] = resp_json
                    clean = {k: v for k, v in resp_json.items() if k != 'prompt_token_ids'}
                    if isinstance(clean.get('choices'), list):
                        clean['choices'] = [{k: v for k, v in c.items() if k != 'token_ids'} for c in clean['choices']]
                    out = json.dumps(clean).encode()
            except Exception:
                rec['response_text'] = data.decode('utf-8', 'replace')
            self.server.log(rec)
        self.send_response(status)
        for k, v in resp_headers:
            self.send_header(k, v)
        self.send_header('Content-Length', str(len(out)))
        self.end_headers()
        self.wfile.write(out)


class LoggingProxy(http.server.ThreadingHTTPServer):
    daemon_threads = True

    def __init__(self, port: int, name: str):
        super().__init__(('127.0.0.1', port), _ProxyHandler)
        self.name = name
        self._lock = threading.Lock()
        self._sink = None
        self.n_logged = 0

    def attach(self, path: Path) -> None:
        """Send this slot's chat records to `path` (one sample's requests.jsonl)."""
        with self._lock:
            self._sink, self.n_logged = path, 0

    def detach(self) -> int:
        with self._lock:
            self._sink = None
            return self.n_logged

    def log(self, rec: dict) -> None:
        with self._lock:
            path = self._sink or UNMATCHED_DIR / f'{self.name}.jsonl'
            self.n_logged += self._sink is not None
            with open(path, 'a', encoding='utf-8') as f:
                f.write(json.dumps(rec) + '\n')


def walk_strings(obj, fn, skip_ids=frozenset(), _seen=None, _depth=0) -> int:
    """Apply fn(str) -> str|None to every string reachable from obj (dicts, lists,
    object/pydantic attributes); a non-None result replaces the string. Returns the
    number of replacements. Used to point a model registry at a proxy port."""
    if _seen is None:
        _seen = set()
    if _depth > 8 or id(obj) in _seen or id(obj) in skip_ids:
        return 0
    if obj is None or isinstance(obj, (str, bytes, int, float, bool)) or inspect.ismodule(obj) \
            or inspect.isclass(obj) or inspect.isroutine(obj):
        return 0
    _seen.add(id(obj))
    n = 0

    def visit(container, key, v):
        nonlocal n
        if isinstance(v, str):
            new = fn(v)
            if new is not None and new != v:
                container[key] = new
                n += 1
        else:
            n += walk_strings(v, fn, skip_ids, _seen, _depth + 1)

    if isinstance(obj, dict):
        for k, v in list(obj.items()):
            visit(obj, k, v)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            visit(obj, i, v)
    elif isinstance(obj, (tuple, set, frozenset)):
        for v in obj:
            if not isinstance(v, str):
                n += walk_strings(v, fn, skip_ids, _seen, _depth + 1)
    else:
        for attr in ('__dict__', '__pydantic_private__', '__pydantic_extra__'):
            d = getattr(obj, attr, None)
            if isinstance(d, dict):
                n += walk_strings(d, fn, skip_ids, _seen, _depth + 1)
    return n


def _http_json(port: int, method: str, path: str, payload=None):
    conn = http.client.HTTPConnection('127.0.0.1', port, timeout=900)
    conn.request(method, path, body=json.dumps(payload) if payload is not None else None,
                 headers={'Content-Type': 'application/json'})
    r = conn.getresponse()
    status, data = r.status, r.read()
    conn.close()
    return status, json.loads(data)


server_instance = None
SLOTS = []
if not SKIP_SAMPLING:
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
    tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    vllm_cfg = VllmConfig(
        model=str(MODEL_PATH),
        port=VLLM_PORT,
        host='127.0.0.1',
        tool_call_parser='gemma4',
        reasoning_parser='gemma4',
        max_model_len=32768,
        dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
        gpu_memory_utilization=0.90,
        enable_auto_tool_choice=True,
        enable_lora=bool(adapter_map),
        max_loras=max(1, len(adapter_map)),
        max_lora_rank=max(detected_ranks) if detected_ranks else 128,
        scheduler_reserve_full_isl=False,
        tensor_parallel_size=tp_size,
        startup_timeout=60 * 20,
        extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
    )
    server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
    server_instance.start()
    print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

    # One model registry per slot, each pointed at its own proxy port.
    _old = f'127.0.0.1:{VLLM_PORT}'
    for i in range(CONCURRENCY):
        port = PROXY_BASE_PORT + i
        proxy = LoggingProxy(port, f'slot{i}')
        threading.Thread(target=proxy.serve_forever, daemon=True).start()
        reg = server_instance.create_model_registry(
            aliases=[declared_model], model_prefix='openai/', api_key=INFERENCE_API_KEY,
        )
        n = walk_strings(reg, lambda s, port=port: s.replace(_old, f'127.0.0.1:{port}') if _old in s else None,
                         skip_ids={id(server_instance)})
        assert n > 0, f'no api_base containing {_old!r} found in the model registry -- cannot route through the proxy'
        SLOTS.append({'i': i, 'port': port, 'proxy': proxy, 'models': reg,
                      'results_dir': WORKING_DIR / 'rft_slots' / f'slot{i}'})
    # registries must not share objects, or slots would log into each other's samples
    for s in SLOTS:
        seen = []
        walk_strings(s['models'], lambda v: seen.append(v) if '127.0.0.1:' in v else None, skip_ids={id(server_instance)})
        ports = set(re.findall(r'127\.0\.0\.1:(\d+)', ' '.join(seen)))
        assert ports == {str(s['port'])}, f"slot{s['i']} registry points at ports {ports}, expected {{{s['port']}}}"

    # Smoke test: the proxy relays, vLLM returns token ids, the client never sees them.
    _st, _models = _http_json(PROXY_BASE_PORT, 'GET', '/v1/models')
    SERVED_MODEL = _models['data'][0]['id']
    _smoke = SAMPLES_DIR / '_smoke.jsonl'
    _smoke.unlink(missing_ok=True)
    SLOTS[0]['proxy'].attach(_smoke)
    _st, _client_view = _http_json(PROXY_BASE_PORT, 'POST', '/v1/chat/completions', {
        'model': SERVED_MODEL, 'max_tokens': 16,
        'messages': [{'role': 'user', 'content': 'Reply with the single word: ok. Time allowance: '
                      f'{float(SAMPLING_MAX_TIME_MINUTES)} minutes'}],
    })
    SLOTS[0]['proxy'].detach()
    _rec = json.loads(_smoke.read_text(encoding='utf-8').splitlines()[-1])
    assert _st == 200, f'smoke request failed: {_client_view}'
    _resp = _rec['response']
    if not _resp.get('prompt_token_ids') or not _resp['choices'][0].get('token_ids'):
        raise RuntimeError('vLLM did not return prompt_token_ids / token_ids for return_token_ids=True. '
                           f'Response keys: {sorted(_resp)}; choice keys: {sorted(_resp["choices"][0])}')
    assert 'prompt_token_ids' not in _client_view, 'token ids leaked to the client'
    assert _rec['rewrites'] == 1 and f'{float(PROMPT_TIME_MINUTES)} minutes' in _rec['request']['messages'][0]['content']
    print(f'proxy OK: {len(SLOTS)} slots on ports {PROXY_BASE_PORT}..{PROXY_BASE_PORT + len(SLOTS) - 1}, '
          f'served model {SERVED_MODEL!r}, smoke prompt {len(_resp["prompt_token_ids"])} tokens -> '
          f'{len(_resp["choices"][0]["token_ids"])} generated')
else:
    print('SKIP_SAMPLING: vLLM not started')

## 5. On-Policy Sampling (adaptive K)

每条样本完成后就落盘到 `rft_samples_<RUN_NAME>/<task>/k<k>/`，续跑时会跳过已完成的样本。

In [ ]:
import asyncio
import concurrent.futures
import traceback

from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator


def _maybe(obj, *names):
    """First non-None attribute among `names` -- harness result field names may vary."""
    for n in names:
        v = getattr(obj, n, None)
        if v is not None:
            return v
    return None


def sample_dir(tid: str, k: int) -> Path:
    return SAMPLES_DIR / tid / f'k{k}'


def load_metas(tid: str) -> list[dict]:
    """Finished samples of a task (a sample is finished once its meta.json exists)."""
    return sorted((json.loads(p.read_text(encoding='utf-8')) for p in (SAMPLES_DIR / tid).glob('k*/meta.json')),
                  key=lambda m: m['k'])


def restore_samples(src) -> int:
    """Copy finished samples from a previous version's output (dirs or zips)."""
    src, n = Path(src), 0
    roots = [src]
    for z in src.rglob('rft_samples_*.zip'):
        dst = Path('/tmp/rft_resume') / z.stem  # not under /kaggle/working: keep it out of the output
        if not dst.exists():
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dst)
        roots.append(dst)
    for root in roots:
        for meta in root.rglob('meta.json'):
            d = meta.parent
            if not re.fullmatch(r'k\d+', d.name) or d.parent.name not in TASK_BY_ID:
                continue
            target = SAMPLES_DIR / d.parent.name / d.name
            if not target.exists():
                shutil.copytree(d, target)
                n += 1
    return n


if RESUME_DIR:
    print(f'restored {restore_samples(RESUME_DIR)} finished samples from {RESUME_DIR}')

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
verification_timeout_seconds = int(eval_section.get('verification_timeout_seconds', 300))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None


def make_evaluator(results_dir: Path, models) -> Evaluator:
    """Same EvalConfig as the dev-eval notebook, except the sampling budgets."""
    results_dir.mkdir(parents=True, exist_ok=True)
    return Evaluator(EvalConfig(
        tasks_path=TASKS_PATH,
        snapshots_dir=SNAPSHOTS_DIR,
        results_dir=results_dir,
        submission_dir=AGENT_DIR,
        models=models,
        sandbox='subprocess',
        timeout_seconds=timeout_seconds,
        verification_timeout_seconds=verification_timeout_seconds,
        max_time_minutes=float(SAMPLING_MAX_TIME_MINUTES),
        max_tool_calls=MAX_TOOL_CALLS,
        max_turns=max_turns,
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15,
            overlap_size=2,
            token_threshold=14336,
            event_retention_size=5,
        ),
        graph_dir=GRAPH_DIR,
        embeddings_dir=EMBEDDINGS_DIR,
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
    ))


ABORT = []          # a message here stops every worker
_JOB_LOCK = threading.Lock()


def run_sample(slot: dict, tid: str, k: int) -> None:
    """One agent session + Phase-2 verification; everything lands in samples/<tid>/k<k>/."""
    task = TASK_BY_ID[tid]
    final = sample_dir(tid, k)
    tmp = final.with_name(final.name + '.partial')
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    rd = slot['results_dir']
    trace_src, log_src = rd / 'traces' / f'trace_{tid}.json', rd / 'logs' / f'{tid}.log'
    for p in (trace_src, log_src):  # left over from an earlier sample of this task in this slot
        p.unlink(missing_ok=True)

    slot['proxy'].attach(tmp / 'requests.jsonl')
    t0, result, crash = time.time(), None, ''
    try:
        res = slot['evaluator'].evaluate_task(task=task, task_index=k + 1, total_tasks=K_FIRST + K_EXTRA)
        result = asyncio.run(res) if asyncio.iscoroutine(res) else res
    except Exception as e:  # one broken sample must not end a multi-hour run
        crash = f'{type(e).__name__}: {e}'
        traceback.print_exc()
    finally:
        n_requests = slot['proxy'].detach()
    wall = time.time() - t0

    patch = str(_maybe(result, 'agent_patch', 'patch') or '').strip()
    has_patch = bool(patch) and patch != 'NO_PATCH'
    if has_patch:
        (tmp / 'patch.diff').write_text(patch, encoding='utf-8')
    test_log = _maybe(result, 'test_output', 'test_log', 'verification_output', 'pytest_output')
    if test_log:
        (tmp / 'test_output.log').write_text(str(test_log), encoding='utf-8')
    for src, name in ((trace_src, 'trace.json'), (log_src, 'agent.log')):
        if src.exists():
            shutil.move(str(src), tmp / name)

    meta = {
        'task_id': tid, 'k': k, 'repo': task.repo, 'slot': slot['i'],
        'resolved': bool(_maybe(result, 'resolved') or False),
        'test_exit_code': _maybe(result, 'test_exit_code', 'exit_code'),
        'patch_chars': len(patch) if has_patch else 0,
        'loc': localize(RAW_TASKS[tid]['patch'], patch if has_patch else '').label,
        'tool_calls': _maybe(result, 'tool_calls'),
        'duration_seconds': _maybe(result, 'duration_seconds', 'duration'),
        'wall_seconds': wall,
        'n_requests': n_requests,
        'error': str(_maybe(result, 'error', 'agent_error') or crash or ''),
        'temperature': SAMPLE_TEMPERATURE, 'top_p': SAMPLE_TOP_P,
        'prompt_time_minutes': PROMPT_TIME_MINUTES,
        'submission_fingerprint': SUBMISSION_FINGERPRINT,
        'finished_at_utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()),
    }
    (tmp / 'meta.json').write_text(json.dumps(meta, indent=2, default=str), encoding='utf-8')
    shutil.rmtree(final, ignore_errors=True)
    tmp.rename(final)
    elapsed_h = (time.time() - NOTEBOOK_START) / 3600
    print(f"[slot{slot['i']} t={elapsed_h:.2f}h] {tid}/k{k} resolved={meta['resolved']} loc={meta['loc']} "
          f"tool_calls={meta['tool_calls']} requests={n_requests} agent={meta['duration_seconds'] or 0:.0f}s "
          f"wall={wall:.0f}s" + (f" error={meta['error'][:120]}" if meta['error'] else ''), flush=True)
    if n_requests == 0 and (meta['tool_calls'] or 0) > 0:
        ABORT.append(f'{tid}/k{k}: the agent made {meta["tool_calls"]} tool calls but no request reached the '
                     'logging proxy -- the harness is not using the retargeted model registry.')


def run_jobs(jobs: list[tuple[str, int]], label: str) -> None:
    """Run (task, k) jobs on all slots. Two samples of one task never run at the
    same time (sandbox dirs and trace file names are keyed by task id)."""
    pending = [j for j in jobs if not (sample_dir(*j) / 'meta.json').exists()]
    active = set()
    print(f'== {label}: {len(pending)} samples to run ({len(jobs) - len(pending)} already done)', flush=True)

    def next_job():
        with _JOB_LOCK:
            if not pending:
                return None
            for i, (tid, k) in enumerate(pending):
                if tid not in active:
                    active.add(tid)
                    return pending.pop(i)
            return 'wait'

    def worker(slot):
        while not ABORT:
            if (time.time() - NOTEBOOK_START) / 3600 > STOP_NEW_SAMPLES_AFTER_HOURS:
                return
            job = next_job()
            if job is None:
                return
            if job == 'wait':
                time.sleep(5)
                continue
            try:
                run_sample(slot, *job)
            except Exception:
                traceback.print_exc()
            finally:
                with _JOB_LOCK:
                    active.discard(job[0])

    with concurrent.futures.ThreadPoolExecutor(len(SLOTS)) as ex:
        list(ex.map(worker, SLOTS))
    if ABORT:
        raise RuntimeError(ABORT[0])
    if pending:
        print(f'!! {len(pending)} samples not started (STOP_NEW_SAMPLES_AFTER_HOURS={STOP_NEW_SAMPLES_AFTER_HOURS}); '
              'attach this output as RESUME_DIR and run again to finish them.')


def extra_reason(tid: str) -> str | None:
    """Adaptive K (TASK.md §2.1), from the first-round samples only."""
    first = [m for m in load_metas(tid) if m['k'] < K_FIRST]
    if len(first) < K_FIRST:
        return None
    n_pass = sum(m['resolved'] for m in first)
    if n_pass == K_FIRST:
        return 'all_pass'
    if n_pass == 0 and any(m['loc'] in ('same_lines', 'same_region') for m in first):
        return 'zero_pass_near_gold'
    return None


def print_pass_table() -> None:
    dist = Counter()
    for tid in TRAIN_IDS:
        ms = load_metas(tid)
        dist[f"{sum(m['resolved'] for m in ms)}/{len(ms)}"] += 1
    print('pass/n -> #tasks:', dict(sorted(dist.items())))


if not SKIP_SAMPLING:
    for s in SLOTS:
        s['evaluator'] = make_evaluator(s['results_dir'], s['models'])
    # round 1: k=0 for every task, then k=1, ... (keeps the two samples of a task apart)
    run_jobs([(tid, k) for k in range(K_FIRST) for tid in TRAIN_IDS], f'round 1 (K={K_FIRST})')
    print_pass_table()
    EXTRA = {tid: why for tid in TRAIN_IDS if (why := extra_reason(tid))}
    print(f'round 2: {len(EXTRA)} tasks get {K_EXTRA} more samples:', dict(Counter(EXTRA.values())))
    run_jobs([(tid, k) for k in range(K_FIRST, K_FIRST + K_EXTRA) for tid in EXTRA], f'round 2 (+{K_EXTRA})')
    print_pass_table()
    server_instance.stop()  # GPU no longer needed; the rest is CPU
    print(f'sampling done at t={(time.time() - NOTEBOOK_START) / 3600:.2f}h')
else:
    print_pass_table()

## 6. Efficient-Pass Filter + Trajectory Choice


In [ ]:
import statistics as st

# TASK.md §2.2-2.3 (revised): any task with an efficient trusted pass qualifies, not only
# sometimes-solved ones; all-pass tasks must also stay within ALL_PASS_CALLS_LIMIT.


def pick_samples(status: str, samples: list[dict]) -> tuple[list[dict], str | None]:
    """Passing samples that qualify, best first (fewest calls), capped per task; or the
    filter that removed the last candidate."""
    steps = (
        ('passing samples have no token ids', lambda s: s['has_ids']),
        (f'bad_loc (no pass in {"/".join(GOOD_LOCS)})', lambda s: s['loc'] in GOOD_LOCS),
        (f'no pass edits by call {EDIT_CALL_LIMIT}',
         lambda s: s['first_edit_call'] is not None and s['first_edit_call'] <= EDIT_CALL_LIMIT),
        (f'too_many_calls (all-pass, > {ALL_PASS_CALLS_LIMIT})',
         lambda s: status != 'all_pass' or s['calls'] <= ALL_PASS_CALLS_LIMIT),
        ('no loss window (no gold read / edit before submit)', lambda s: bool(s['loss_window'])),
    )
    cands = [s for s in samples if s['resolved']]
    for why, ok in steps:
        cands = [s for s in cands if ok(s)]
        if not cands:
            return [], why
    cap = MAX_PER_TASK_ALL_PASS if status == 'all_pass' else MAX_PER_TASK_SOMETIMES
    return sorted(cands, key=lambda s: (s['calls'], s['completion_tokens'], s['k']))[:cap], None


SELECTION = []   # one row per train task -> selection.json
for tid in TRAIN_IDS:
    metas = load_metas(tid)
    gold = gold_files(tid)
    samples = []
    for m in metas:
        turns = load_turns(sample_dir(tid, m['k']) / 'requests.jsonl', with_ids=False)
        a = analyze_turns(turns, gold)
        win = loss_window(turns, a)
        samples.append({'k': m['k'], 'resolved': m['resolved'], 'loc': m['loc'],
                        'harness_tool_calls': m['tool_calls'], **a, 'loss_window': win})
    n, n_pass = len(samples), sum(s['resolved'] for s in samples)
    status = ('not_sampled' if n == 0 else 'all_fail' if n_pass == 0 else 'all_pass' if n_pass == n else 'sometimes')
    picked, reason = pick_samples(status, samples) if n_pass else ([], None)
    SELECTION.append({'task_id': tid, 'repo': RAW_TASKS[tid]['repo'], 'n': n, 'n_pass': n_pass, 'status': status,
                      'gold_files': gold, 'samples': samples, 'chosen_ks': [s['k'] for s in picked], 'reason': reason})

status_counts = Counter(r['status'] for r in SELECTION)
CHOSEN = [(r, k) for r in SELECTION for k in r['chosen_ks']]   # one entry per chosen trajectory
_chosen_tasks = [r for r in SELECTION if r['chosen_ks']]
print('task status:', dict(status_counts))
print(f'chosen: {len(CHOSEN)} trajectories over {len(_chosen_tasks)} tasks',
      dict(Counter(r['status'] for r, _ in CHOSEN)))
for r in SELECTION:
    if r['reason']:
        print(f"  dropped {r['task_id']} ({r['status']} {r['n_pass']}/{r['n']}): {r['reason']}")

# id cross-check: proxy-counted calls vs the harness's own count
_mismatch = [(r['task_id'], s['k'], s['calls'], s['harness_tool_calls']) for r in SELECTION for s in r['samples']
             if s['harness_tool_calls'] is not None and s['calls'] != s['harness_tool_calls']]
print(f'proxy vs harness tool-call count mismatches: {len(_mismatch)}', _mismatch[:10])

_all = [s for r in SELECTION for s in r['samples']]
_fe = lambda ss: st.median(x['first_edit_call'] for x in ss if x['first_edit_call']) if any(x['first_edit_call'] for x in ss) else None
_chosen_samples = [next(s for s in r['samples'] if s['k'] == k) for r, k in CHOSEN]
print(f'median edit@call: all samples {_fe(_all)}, passing {_fe([s for s in _all if s["resolved"]])}, '
      f'chosen {_fe(_chosen_samples)}')
print(f"{'task':<16}{'status':>10}{'pass':>6}{'k':>3}{'calls':>6}{'edit@call':>10}{'gold@turn':>10}{'loss window':>13}")
for (r, k), s in zip(CHOSEN, _chosen_samples):
    print(f"{r['task_id']:<16}{r['status']:>10}{r['n_pass']:>3}/{r['n']:<2}{k:>3}{s['calls']:>6}{s['first_edit_call']:>10}"
          f"{str(s['gold_read_turn']):>10}{str(s['loss_window']):>13}")


### 6b. Self-Sampling Audit

只打印、不过滤（`GOOD_LOCS` 已经在上一节起作用），结果写入 `stats.json` 的 `audit`：
1. **标签可信度**：通过但补丁没碰参考文件的样本。
2. **分布对照**：采样温度下的行为是否与 submission 温度下的 dev60 基线接近。
3. **时间残留**：在第 7 节统计。模型是否在 `get_status` 里看到采样用的 60 分钟墙钟。
4. **计数差异**：选中样本里 proxy 与 harness 的调用计数是否一致。


In [ ]:
AUDIT = {}
# dev60 baseline at the submission temperature (TASK.md §0): first-edit median, tasks using all 28 calls
DEV60_BASELINE = {'median_first_edit_call': 19, 'budget_exhausted': '31/60'}

# 1. label trust: passes whose patch does not touch the reference files
_untrusted = [(r['task_id'], s['k'], s['loc']) for r in SELECTION for s in r['samples']
              if s['resolved'] and s['loc'] not in GOOD_LOCS]
AUDIT['untrusted_passes'] = _untrusted
print(f'1. passes outside GOOD_LOCS (never chosen): {len(_untrusted)}')
for tid, locs in sorted(Counter(t for t, _, _ in _untrusted).items()):
    r = next(x for x in SELECTION if x['task_id'] == tid)
    print(f"   {tid}: {locs}/{r['n_pass']} passes", dict(Counter(l for t, _, l in _untrusted if t == tid)),
          '-> target tests may pass without a fix; check with a null-patch run' if all(
              s['loc'] in ('no_edit', 'scratch_only') for s in r['samples'] if s['resolved']) else '')

# 2. behaviour at the sampling temperature vs the dev60 baseline
_used = lambda s: s['harness_tool_calls'] if s['harness_tool_calls'] is not None else s['calls']
_exhausted = sum(_used(s) >= MAX_TOOL_CALLS for s in _all)
AUDIT['distribution'] = {
    'sampling_temperature': SAMPLE_TEMPERATURE,
    'submission_temperature': (SUBMISSION_SAMPLING or {}).get('temperature'),
    'median_first_edit_call': _fe(_all),
    'median_calls': st.median(s['calls'] for s in _all) if _all else None,
    'budget_exhausted': f'{_exhausted}/{len(_all)}',
    'dev60_baseline': DEV60_BASELINE,
}
print('2. train pool vs dev60 baseline:', AUDIT['distribution'])

# 4. call counts of the chosen samples (edit@call uses the proxy count)
AUDIT['chosen_count_mismatch'] = [(r['task_id'], k, s['calls'], s['harness_tool_calls'])
                                  for (r, k), s in zip(CHOSEN, _chosen_samples)
                                  if s['harness_tool_calls'] is not None and s['calls'] != s['harness_tool_calls']]
print(f"4. chosen samples with proxy != harness call count: {AUDIT['chosen_count_mismatch']}")


## 7. Key-Turn Mask + Pack Sequences + Chat-Template Check


In [ ]:
from transformers import AutoTokenizer

# Same tokenizer as the bf16 training base (the w4a16 checkpoint ships it unchanged).
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH))
END_IDS = set()
_eos = tokenizer.eos_token_id
END_IDS.update(_eos if isinstance(_eos, list) else [_eos] if _eos is not None else [])
try:
    from transformers import GenerationConfig
    _g = GenerationConfig.from_pretrained(str(MODEL_PATH)).eos_token_id
    END_IDS.update(_g if isinstance(_g, list) else [_g] if _g is not None else [])
except Exception:
    pass
for _tok in ('<turn|>', '<end_of_turn>', '<eos>'):
    _id = tokenizer.convert_tokens_to_ids(_tok)
    if _id is not None and _id != tokenizer.unk_token_id:
        END_IDS.add(_id)
print('end-of-turn ids:', {i: tokenizer.convert_ids_to_tokens(i) for i in sorted(END_IDS)})

TRAIN_ROWS, READABLE = [], []
PACK_STATS, TEMPLATE, MASK_STATS, TIME_AUDIT = Counter(), Counter(), Counter(), Counter()
TRAJ_INFO = {}   # (task_id, k) -> loss window / key turns / masked turns, for selection.json + report.md
TEMPLATE_MISMATCHES = []


def check_template(tid: str, turns: list[dict]) -> None:
    """TASK.md §2.1: vLLM's rendered prompt must equal tokenizer.apply_chat_template, token for token."""
    for t in turns:
        if t['superseded'] or not t['has_ids'] or not t['request']:
            continue
        try:
            ids = render_prompt_ids(tokenizer, t['request'])
        except Exception as e:
            TEMPLATE['render_error'] += 1
            if len(TEMPLATE_MISMATCHES) < 5:
                TEMPLATE_MISMATCHES.append({'task_id': tid, 'turn': t['idx'], 'error': f'{type(e).__name__}: {e}'})
            continue
        if ids == t['prompt_ids']:
            TEMPLATE['match'] += 1
        else:
            TEMPLATE['mismatch'] += 1
            if len(TEMPLATE_MISMATCHES) < 5:
                j = next((i for i, (x, y) in enumerate(zip(ids, t['prompt_ids'])) if x != y), min(len(ids), len(t['prompt_ids'])))
                TEMPLATE_MISMATCHES.append({
                    'task_id': tid, 'turn': t['idx'], 'first_diff': j, 'len_template': len(ids), 'len_vllm': len(t['prompt_ids']),
                    'template': tokenizer.decode(ids[max(0, j - 40):j + 40]),
                    'vllm': tokenizer.decode(t['prompt_ids'][max(0, j - 40):j + 40]),
                })


_inflated_s = PROMPT_TIME_MINUTES * 60 if PROMPT_TIME_MINUTES else None
for r, k in CHOSEN:
    tid = r['task_id']
    turns = load_turns(sample_dir(tid, k) / 'requests.jsonl', with_ids=True)
    PACK_STATS['results_unmatched'] += attach_results(turns)
    a = analyze_turns(turns, r['gold_files'])
    start, end = loss_window(turns, a)
    selected, reasons = key_turns(turns, a, patch_files(tid, k))
    MASK_STATS.update(reasons.values())
    TRAJ_INFO[(tid, k)] = {'loss_window': [start, end], 'n_window_turns': len(selected) + len(reasons),
                           'key_turns': sorted(selected), 'masked': {str(i): w for i, w in sorted(reasons.items())}}
    # audit 3: a get_status result at turn t is seen from turn t+1 on; flag the sampling-time
    # wall clock (SAMPLING_MAX_TIME_MINUTES, not rewritten in time_seconds_remaining)
    _status = [(t['idx'], res) for t in turns if not t['superseded']
               for (fn, _), res in zip(t['tool_calls'], t['results'] or []) if fn == 'get_status']
    TIME_AUDIT['get_status_calls'] += len(_status)
    _inflated = [i for i, res in _status if _inflated_s and isinstance(res, dict)
                 and (res.get('time_seconds_remaining') or 0) > _inflated_s]
    if _inflated:
        TIME_AUDIT['trajectories_seeing_inflated_time'] += 1
        TIME_AUDIT['loss_turns_after_inflated_time'] += sum(i > _inflated[0] for i in selected)
    PACK_STATS['loss_turns_truncated_output'] += sum(1 for t in turns if start <= t['idx'] <= end
                                                     and not t['superseded'] and t['finish_reason'] == 'length')
    segs, stats = pack_turns(turns, selected, END_IDS, MAX_SEQ_LEN)
    PACK_STATS.update(stats)
    check_template(tid, turns)

    for si, seg in enumerate(segs):
        n_loss = sum(l != -100 for l in seg['labels'])
        TRAIN_ROWS.append({
            'id': f'{tid}_k{k}_s{si}', 'task_id': tid, 'sample': k, 'segment': si,
            'input_ids': seg['input_ids'], 'labels': seg['labels'],
            'n_tokens': len(seg['input_ids']), 'n_loss_tokens': n_loss, 'n_loss_turns': len(seg['loss_turns']),
            'turns': seg['turns'], 'loss_turns': seg['loss_turns'],
        })
    READABLE.append({
        'task_id': tid, 'sample': k, 'repo': r['repo'], 'loss_window': [start, end],
        'turns': [{
            'idx': t['idx'], 'train': t['idx'] in selected, 'mask_reason': reasons.get(t['idx']),
            'superseded': t['superseded'],
            'prompt_tokens': t['n_prompt'], 'completion_tokens': t['n_completion'], 'finish_reason': t['finish_reason'],
            'reasoning': t['reasoning'], 'content': t['content'],
            'tool_calls': [{'name': fn, 'arguments': args} for fn, args in t['tool_calls']],
        } for t in turns],
    })
    del turns

if not TEMPLATE:  # nothing chosen (e.g. a small trial run): still check the template on one sample
    _first = next((m for tid in TRAIN_IDS for m in load_metas(tid)), None)
    if _first:
        check_template(_first['task_id'], load_turns(sample_dir(_first['task_id'], _first['k']) / 'requests.jsonl'))

n_tok =sum(r['n_tokens'] for r in TRAIN_ROWS)
n_loss_tok = sum(r['n_loss_tokens'] for r in TRAIN_ROWS)
n_loss_turns = sum(r['n_loss_turns'] for r in TRAIN_ROWS)
_tmpl_total = sum(TEMPLATE.values())
AUDIT['inflated_time'] = dict(TIME_AUDIT)
_n_window = sum(v['n_window_turns'] for v in TRAJ_INFO.values())
print(f'loss turns: {n_loss_turns} key of {_n_window} in the loss windows; masked:', dict(MASK_STATS))
print(f'sequences: {len(TRAIN_ROWS)}, tokens: {n_tok / 1e6:.2f}M, loss tokens: {n_loss_tok / 1e3:.0f}k')
# chain_break is expected: each history re-renders the earlier generations, so a turn's prompt
# never extends the previous prompt + generation and every loss turn is its own sequence.
print('packing:', dict(PACK_STATS))
print('3. audit, sampling wall clock seen in get_status:', AUDIT['inflated_time'])
print(f"template check: {TEMPLATE['match']}/{_tmpl_total} prompts identical to apply_chat_template", dict(TEMPLATE))
for mm in TEMPLATE_MISMATCHES:
    print('  mismatch:', json.dumps(mm, ensure_ascii=False)[:800])

# Eyeball: the labelled spans of the first sequence should read as
# <|channel>thought ... <channel|> ... <|tool_call> ... (+ end-of-turn token).
if TRAIN_ROWS:
    _r = TRAIN_ROWS[0]
    _spans, _s = [], None
    for i, l in enumerate(_r['labels'] + [-100]):
        if l != -100 and _s is None:
            _s = i
        elif l == -100 and _s is not None:
            _spans.append((_s, i))
            _s = None
    print(f"{_r['id']}: {len(_spans)} labelled spans")
    for s, e in _spans[:2]:
        txt = tokenizer.decode(_r['input_ids'][s:e])
        print(f'--- span [{s}, {e}) ---\n{txt[:600]}\n  [...]\n{txt[-300:]}')

## 8. Write Outputs for Download

In [ ]:
if DATA_OUT.exists():
    shutil.rmtree(DATA_OUT)
DATA_OUT.mkdir(parents=True)


def _write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open('w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


for r in SELECTION:
    r['key_turns'] = {str(k): TRAJ_INFO[(r['task_id'], k)] for k in r['chosen_ks'] if (r['task_id'], k) in TRAJ_INFO}

_write_jsonl(DATA_OUT / 'train.jsonl', TRAIN_ROWS)
_write_jsonl(DATA_OUT / 'train_readable.jsonl', READABLE)
(DATA_OUT / 'selection.json').write_text(json.dumps(SELECTION, indent=2, default=str), encoding='utf-8')

_n_samples = sum(r['n'] for r in SELECTION)
STATS = {
    'run_name': RUN_NAME,
    'created_at_utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()),
    'model': declared_model,
    'tokenizer_path': str(MODEL_PATH),
    'submission_fingerprint': SUBMISSION_FINGERPRINT,
    'submission_sampling': SUBMISSION_SAMPLING,
    'sampling': {'temperature': SAMPLE_TEMPERATURE, 'top_p': SAMPLE_TOP_P, 'max_tool_calls': MAX_TOOL_CALLS,
                 'harness_max_time_minutes': SAMPLING_MAX_TIME_MINUTES, 'prompt_time_minutes': PROMPT_TIME_MINUTES,
                 'k_first': K_FIRST, 'k_extra': K_EXTRA, 'concurrency': CONCURRENCY},
    'selection': {'edit_call_limit': EDIT_CALL_LIMIT, 'all_pass_calls_limit': ALL_PASS_CALLS_LIMIT,
                  'max_per_task_all_pass': MAX_PER_TASK_ALL_PASS, 'max_per_task_sometimes': MAX_PER_TASK_SOMETIMES,
                  'good_locs': list(GOOD_LOCS), 'post_edit_keep': POST_EDIT_KEEP, 'max_seq_len': MAX_SEQ_LEN},
    'train_ids': TRAIN_IDS,
    'n_samples': _n_samples,
    'n_resolved_samples': sum(r['n_pass'] for r in SELECTION),
    'task_status': dict(status_counts),
    'n_tasks_chosen': len(_chosen_tasks),
    'n_trajectories': len(CHOSEN),
    'chosen_by_status': dict(Counter(r['status'] for r, _ in CHOSEN)),
    'n_sequences': len(TRAIN_ROWS),
    'n_window_turns': _n_window,
    'n_loss_turns': n_loss_turns,
    'masked_turns': dict(MASK_STATS),
    'n_tokens': n_tok,
    'n_loss_tokens': n_loss_tok,
    'max_sequence_tokens': max((r['n_tokens'] for r in TRAIN_ROWS), default=0),
    'packing': dict(PACK_STATS),
    'template_check': dict(TEMPLATE),
    'template_mismatch_examples': TEMPLATE_MISMATCHES,
    'end_of_turn_ids': sorted(END_IDS),
    'median_edit_at_call': {'all': _fe(_all), 'passing': _fe([s for s in _all if s['resolved']]), 'chosen': _fe(_chosen_samples)},
    'audit': AUDIT,
    'notebook_hours': (time.time() - NOTEBOOK_START) / 3600,
}
(DATA_OUT / 'stats.json').write_text(json.dumps(STATS, indent=2, default=str), encoding='utf-8')

_lines = [
    f'# RFT data `{RUN_NAME}`', '',
    f'- submission `{SUBMISSION_FINGERPRINT}`, model `{declared_model}`, temperature {SAMPLE_TEMPERATURE}, '
    f'top_p {SAMPLE_TOP_P}, max_tool_calls {MAX_TOOL_CALLS}, model told {PROMPT_TIME_MINUTES} min',
    f'- samples: {_n_samples} ({STATS["n_resolved_samples"]} resolved) over {len(TRAIN_IDS)} tasks',
    f'- task status: {dict(status_counts)}',
    f'- selection: {STATS["selection"]}',
    f'- chosen: {len(CHOSEN)} trajectories over {len(_chosen_tasks)} tasks {STATS["chosen_by_status"]}',
    f'- loss turns: {n_loss_turns} key of {_n_window} window turns; masked {dict(MASK_STATS)}',
    f'- sequences: {len(TRAIN_ROWS)} | tokens: {n_tok} | loss tokens: {n_loss_tok}',
    f'- audit: untrusted passes {len(AUDIT["untrusted_passes"])}, {AUDIT["distribution"]}, '
    f'sampling wall clock seen {AUDIT.get("inflated_time")}',
    f'- template check (vLLM prompt ids == apply_chat_template): {dict(TEMPLATE)}',
    f'- packing: {dict(PACK_STATS)}',
    f'- median edit@call: {STATS["median_edit_at_call"]}', '',
    '| task | pass/n | status | chosen k | calls | edit@call | gold read @turn | loss window | key/window turns | masked |',
    '|---|---|---|---|---|---|---|---|---|---|',
]
for r in SELECTION:
    for s in [next(x for x in r['samples'] if x['k'] == k) for k in r['chosen_ks']] or [None]:
        info = TRAJ_INFO.get((r['task_id'], s['k'])) if s else None
        turns_col = f"{len(info['key_turns'])}/{info['n_window_turns']}" if info else ''
        masked_col = ', '.join(f'{w} {c}' for w, c in Counter(info['masked'].values()).most_common()) if info else ''
        _lines.append(
            f"| {r['task_id']} | {r['n_pass']}/{r['n']} | {r['status']}{' (' + r['reason'] + ')' if r['reason'] else ''} | "
            f"{s['k'] if s else ''} | {s['calls'] if s else ''} | {s['first_edit_call'] if s else ''} | "
            f"{s['gold_read_turn'] if s else ''} | {s['loss_window'] if s else ''} | {turns_col} | {masked_col} |")
(DATA_OUT / 'report.md').write_text('\n'.join(_lines) + '\n', encoding='utf-8')
(DATA_OUT / 'README.md').write_text(
    '# Files\n\n'
    '- `train.jsonl`: one training sequence per line. `input_ids` are the exact tokens vLLM saw / generated '
    '(Gemma 4 tokenizer); `labels` = token id on trained positions, -100 elsewhere (already aligned with '
    '`input_ids`, the model shifts internally). Extra keys: id, task_id, sample, segment, n_tokens, '
    'n_loss_tokens, n_loss_turns, turns, loss_turns.\n'
    '- `train_readable.jsonl`: the chosen trajectories as text, `train: true` on loss turns, '
    '`mask_reason` on window turns left out of the loss.\n'
    '- `selection.json`: every task / sample with pass status, budget usage and the selection decision '
    '(`chosen_ks`, `key_turns`).\n'
    '- `stats.json`, `report.md`: run summary.\n'
    '\nLoad: `datasets.load_dataset("json", data_files="train.jsonl")` and keep `input_ids` / `labels`.\n',
    encoding='utf-8',
)

data_zip = Path(shutil.make_archive(str(WORKING_DIR / f'rft_data_{RUN_NAME}'), 'zip',
                                    root_dir=WORKING_DIR, base_dir=DATA_OUT.name))
samples_zip = Path(shutil.make_archive(str(WORKING_DIR / f'rft_samples_{RUN_NAME}'), 'zip',
                                       root_dir=WORKING_DIR, base_dir=SAMPLES_DIR.name))
print(f'{data_zip} ({data_zip.stat().st_size / 1024**2:.1f} MiB)')
print(f'{samples_zip} ({samples_zip.stat().st_size / 1024**2:.1f} MiB)')
print()
print(f"{'metric':<34}{'got':>12}")
for name, got in (
    ('sometimes-solved tasks', status_counts['sometimes']),
    ('tasks chosen', len(_chosen_tasks)),
    ('trajectories', len(CHOSEN)),
    ('loss turns (key / window)', f'{n_loss_turns}/{_n_window}'),
    ('sequences', len(TRAIN_ROWS)),
    ('tokens', f'{n_tok / 1e6:.2f}M'),
    ('template match', f"{TEMPLATE['match']}/{_tmpl_total}"),
):
    print(f'{name:<34}{str(got):>12}')
print(f'\nnotebook time: {(time.time() - NOTEBOOK_START) / 3600:.2f} h. Download {data_zip.name} from the Output tab.')